In [1]:
#%%capture
%run ../dca/input/Format.ipynb
import ROOT as root
from array import array
import math
root.gErrorIgnoreLevel = root.kFatal
%jsroot on
root.gStyle.SetOptStat(0)

/home/yoren/.local/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Welcome to JupyROOT 6.30/06


Error in <TUnixSystem::FindDynamicLibrary>: input/logo/PHENIXTools/lib/libLogoPainter.so does not exist in /home/yoren/bnl/ROOT/install/lib:.:/home/yoren/bnl/ROOT/install/lib:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/lib/x86_64-linux-gnu/tls/haswell/x86_64:/lib/x86_64-linux-gnu/tls/haswell:/lib/x86_64-linux-gnu/tls/x86_64:/lib/x86_64-linux-gnu/tls:/lib/x86_64-linux-gnu/haswell/x86_64:/lib/x86_64-linux-gnu/haswell:/lib/x86_64-linux-gnu/x86_64:/lib/x86_64-linux-gnu:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/usr/lib/x86_64-linux-gnu/tls/haswell/x86_64:/usr/lib/x86_64-linux-gnu/tls/haswell:/usr/lib/x86_64-linux-gnu/tls/x86_64:/usr/lib/x86_64-linux-gnu/tls:/usr/lib/x86_64-linux-gnu/haswell/x86_64:/usr/lib/x86_64-linux-gnu/haswell:/usr/lib/x86_64-linux-gnu/x86_64:/usr/lib/x86_64-linux-gnu:/lib/glibc-hwcaps/x86-64-v3:/lib/glibc-hwcaps/x86-64-v2:/lib/tls/haswell/x86_64:/lib/tls/haswell:/lib

In [2]:
sim_pt_file_names = ['pi0_gee_v05_pa.root']
hists_pt_sim = []
hist_select_3D_names = ["hist_pt_mother_weight","inv_mass_dca_gen_5","inv_mass_dca_gen_6","inv_mass_dca_gen_7","inv_mass_dca_gen_8","inv_mass_dca_gen_9"]
sim_file_path = "../sim/output/spring26/dca/"
for iFile in range(len(sim_pt_file_names)):
    infile = root.TFile.Open(sim_file_path+sim_pt_file_names[iFile], "read")
    hists_pt_sim_file = []
    for ihist in range(len(hist_select_3D_names)):
        hists_pt_sim_file.append(infile.Get(hist_select_3D_names[ihist]))
        hists_pt_sim_file[-1].SetDirectory(root.nullptr)
        hists_pt_sim_file[-1].SetName(hists_pt_sim_file[-1].GetName()+f"_sim_{iFile}")
    hists_pt_sim.append(hists_pt_sim_file)
    infile.Close()

In [3]:
MPI0 = 0.134  # used in your mt shift

def Hagedorn_Yield_Func(x, p):
    pt   = x[0]
    mass = p[0]
    if pt*pt + mass*mass - MPI0*MPI0 < 0:
        return 0.0
    mt   = root.TMath.Sqrt(pt*pt + mass*mass - MPI0*MPI0)

    A  = p[1]
    a  = p[2]
    b  = p[3]
    p0 = p[4]
    n  = p[5]

    return 2.0*root.TMath.Pi() * pt * A * pow(root.TMath.Exp(-a*mt - b*mt*mt) + mt/p0, -n)

def fHagedorn(name, lowerlim, upperlim, mass=0.135, A=504.5, a=0.5169, b=0.1626, p0=0.7366, n=8.274):
    f = root.TF1(name, Hagedorn_Yield_Func, lowerlim, upperlim, 6)
    f.SetParameters(mass, A, a, b, p0, n)
    f.SetParNames("mass","A","a","b","p0","n")
    return f

In [4]:
# ---- arrays used in hadron_yield construction ----
dNdy_pp = [42.2*95/257, 42.2*11/257, 8.6*42/257, 4.3, 4.3, 0.92, 0.93, 0.93, 1.77e-5, 0.133/1000/42.2, 0.12, 0.01, 10]
m_pp    = [0.1349766, 0.547862, 0.77526, 0.78265, 0.78265, 0.95778, 1.019461, 1.019461, 3.096916, 3.68609, -1.019461, -9.46030, 1]

Ncolls  = [292., 771, 282.4, 82.6, 12.1, 3]

Br_to_ee = [1.174e-2, 6.9e-3, 4.73e-5, 7.38e-5, 7.7e-4, 4.73e-4, 2.98e-4, 1.08e-4, 5.94e-2, 7.9e-3, 1, 1, 1]

hith_pt_ratio = [1.0, 0.48, 1.0, 0.78, 0.78, 0.25, 0.4, 0.4, 0.01, 0.5, 1, 1, 1]
pi0_high_pt_ratio = [0.0016569944307690066,0.00623224, 0.00228273, 0.000667683, 9.78082e-05, 2.425e-05]

RAA_jpsi = [0.4,0.2,0.3,0.7,0.8,0.8]

hag_params_try = [
  [ 573.1,  0.4178, 0.2118, 0.7282, 8.287],
  [1505.2,  0.4347, 0.2610, 0.7205, 8.319], # 0–20%
  [735.9,   0.4103, 0.1884, 0.7316, 8.272], # 20–40%
  [296.2,   0.4038, 0.1290, 0.7313, 8.207], # 40–60%
  [118.1,   0.4416, 0.0559, 0.7230, 8.163], # 60–80%
  [51.1,    0.2470, 0.0619, 0.7101, 8.459], # 80–93%
]

In [5]:
# single “centrality-free” choice
part = 0 
i0 = 2  # use the first row consistently

hadron_yield = fHagedorn("hadron_yield", 0.01, 25, mass=m_pp[part])

print("integral pre-scale:", hadron_yield.Integral(0.01, 20))

# default scaling: A *= 1.0 * Br_to_ee[part]
A0 = hadron_yield.GetParameter(1)
hadron_yield.SetParameter(1, A0 * 1.0 * Br_to_ee[part])

# if (part<2 || part==8): replace params with hag_params_try[i0], then A *= Br_to_ee[part]
if (part < 2) or (part == 8):
    c,a,b,p0,n = hag_params_try[i0]
    hadron_yield = fHagedorn("hadron_yield_new", 0.01, 25, mass=m_pp[part], A=c, a=a, b=b, p0=p0, n=n)
    A1 = hadron_yield.GetParameter(1)
    hadron_yield.SetParameter(1, A1 * Br_to_ee[part])

# if (part==8 || part==9): normalize integral and scale with dNdy * Br * RAA
# (since no Ncolls, we drop it; since no i, use RAA_jpsi[i0])
if (part == 8) or (part == 9):
    integ = hadron_yield.Integral(0.01, 10.0)
    if integ > 0:
        A = hadron_yield.GetParameter(1)
        hadron_yield.SetParameter(1, A / integ * dNdy_pp[part] * Br_to_ee[part] * RAA_jpsi[i0] * Ncolls[i0])

# if (part < 8 && part > 0): high-pt ratio scaling (use i0)
if (part < 8) and (part > 0):
    denom = (hadron_yield.Eval(5.0) / Br_to_ee[part] / pi0_high_pt_ratio[i0])
    if denom != 0:
        A = hadron_yield.GetParameter(1)
        hadron_yield.SetParameter(1, A * hith_pt_ratio[part] / denom)

hadron_yield.SetNpx(100000)
print("integral post-scale:", hadron_yield.Integral(0.01, 25) / Br_to_ee[part])
print("eval(5)/BR:", hadron_yield.Eval(5.0) / Br_to_ee[part])

integral pre-scale: 95.19739336423685
integral post-scale: 116.98798170143553
eval(5)/BR: 0.0028756339312055023


In [6]:
hists_pt_sim_proj = hists_pt_sim[0][0].ProjectionX("hists_pt_sim_proj",21,40)
hists_pt_sim_proj.Scale(10./hists_pt_sim[0][0].GetEntries()*93/20); hists_pt_sim_proj.Scale(1./hists_pt_sim_proj.GetBinWidth(1))

In [7]:
hists_pt_sim_proj_ee = hists_pt_sim[0][1].ProjectionZ("hists_pt_sim_proj_ee",1,200,1,80)
hists_pt_sim_proj_ee.Scale(10.0 /hists_pt_sim[0][1].GetEntries()/Ncolls[i0]*291); hists_pt_sim_proj_ee.Scale(1./hists_pt_sim_proj_ee.GetBinWidth(1))

In [8]:

c11 = root.TCanvas("c11","c11",800,600)
root.gPad.SetLogy()
hists_pt_sim_proj.GetXaxis().SetRangeUser(1e-6, 10)
hists_pt_sim_proj.Draw("HIST")
hadron_yield.Draw("SAME")
hists_pt_sim_proj_ee.SetLineColor(root.kBlack)
hists_pt_sim_proj_ee.Draw("SAME HIST")

c11.Draw()

In [9]:
import ROOT as root
import os

base = "input/phenix"
pattern = "HEPData-ins838580-v1-Figure_30b_pt.{ipt}.root"

# pt.1 ... pt.8 scale labels: 10^5,10^4,...,10^-2 (8 files)
# you want to scale DOWN => multiply by 1/scale
scale_label_by_ipt = {ipt: 10.0**(5-ipt) for ipt in range(1, 9)}  # 1->1e5, 8->1e-2

# --- recursive walker over keys/directories ---
def walk(dir_or_file, prefix=""):
    out = []
    for k in dir_or_file.GetListOfKeys():
        name = k.GetName()
        o = k.ReadObj()
        path = f"{prefix}/{name}" if prefix else name
        out.append((path, o))
        if o.InheritsFrom("TDirectory"):
            out += walk(o, path)
    return out

# --- find a sensible "main" object (prefer Hist1D_y1 / TH1 first, otherwise TGraph) ---
def pick_main_obj(items):
    # strong preference by name
    preferred_names = ["Hist1D_y1", "Graph1D_y1"]
    for want in preferred_names:
        for path, o in items:
            if os.path.basename(path) == want and (o.InheritsFrom("TH1") or o.InheritsFrom("TGraph")):
                return path, o

    # otherwise first TH1/TGraph
    for path, o in items:
        if o.InheritsFrom("TH1") or o.InheritsFrom("TGraph"):
            return path, o

    raise RuntimeError("No TH1 or TGraph found.")

# --- scale object in Y (and its Y-errors if present) ---
def scale_down(obj, factor):  # factor is the ORIGINAL scale (1e5 etc); we apply 1/factor
    inv = 1.0 / float(factor)

    if obj.InheritsFrom("TH1"):
        h = obj.Clone(obj.GetName() + f"_scaledDown_{factor:g}")
        h.SetDirectory(0)
        h.Scale(inv)
        return h

    if obj.InheritsFrom("TGraph"):
        g2 = obj.Clone(obj.GetName() + f"_scaledDown_{factor:g}")
        n = g2.GetN()
        x = root.Double(0.0); y = root.Double(0.0)
        for i in range(n):
            g2.GetPoint(i, x, y)
            g2.SetPoint(i, float(x), float(y) * inv)

        if g2.InheritsFrom("TGraphAsymmErrors"):
            for i in range(n):
                g2.SetPointEYlow(i,  g2.GetErrorYlow(i)  * inv)
                g2.SetPointEYhigh(i, g2.GetErrorYhigh(i) * inv)
        elif g2.InheritsFrom("TGraphErrors"):
            for i in range(n):
                g2.SetPointError(i, g2.GetErrorX(i), g2.GetErrorY(i) * inv)

        return g2

    raise RuntimeError(f"Don't know how to scale: {obj.ClassName()}")

# ---- load + scale all 8 files ----
files = {}
objs_raw = {}
objs_scaled = []  # <- this is the array you want to plot later
meta = []         # optional info per object: (ipt, fname, obj_path, class, scale, applied)

for ipt in range(1, 9):
    fname = f"{base}/{pattern.format(ipt=ipt)}"
    f = root.TFile.Open(fname, "READ")
    if not f or f.IsZombie():
        raise RuntimeError(f"Could not open: {fname}")

    items = walk(f)
    obj_path, obj = pick_main_obj(items)

    scale = scale_label_by_ipt[ipt]
    obj_s = scale_down(obj, scale)

    # keep refs alive
    files[ipt] = f
    objs_raw[ipt] = obj
    objs_scaled.append(obj_s)
    meta.append((ipt, fname, obj_path, obj.ClassName(), scale, 1.0/scale))

print("Loaded & scaled:")
for (ipt, fname, obj_path, cls, scale, applied) in meta:
    print(f" pt.{ipt}: {os.path.basename(fname)}  | {obj_path} ({cls})  | scale label={scale:g}  applied={applied:g}")

# objs_scaled is your array of scaled objects (TH1* or TGraph*)
# If you want it keyed by pt index instead:
objs_scaled_by_ipt = {ipt: objs_scaled[ipt-1] for ipt in range(1, 9)}

Loaded & scaled:
 pt.1: HEPData-ins838580-v1-Figure_30b_pt.1.root  | Figure 30b pt.1/Hist1D_y1 (TH1F)  | scale label=10000  applied=0.0001
 pt.2: HEPData-ins838580-v1-Figure_30b_pt.2.root  | Figure 30b pt.2/Hist1D_y1 (TH1F)  | scale label=1000  applied=0.001
 pt.3: HEPData-ins838580-v1-Figure_30b_pt.3.root  | Figure 30b pt.3/Hist1D_y1 (TH1F)  | scale label=100  applied=0.01
 pt.4: HEPData-ins838580-v1-Figure_30b_pt.4.root  | Figure 30b pt.4/Hist1D_y1 (TH1F)  | scale label=10  applied=0.1
 pt.5: HEPData-ins838580-v1-Figure_30b_pt.5.root  | Figure 30b pt.5/Hist1D_y1 (TH1F)  | scale label=1  applied=1
 pt.6: HEPData-ins838580-v1-Figure_30b_pt.6.root  | Figure 30b pt.6/Hist1D_y1 (TH1F)  | scale label=0.1  applied=10
 pt.7: HEPData-ins838580-v1-Figure_30b_pt.7.root  | Figure 30b pt.7/Hist1D_y1 (TH1F)  | scale label=0.01  applied=100
 pt.8: HEPData-ins838580-v1-Figure_30b_pt.8.root  | Figure 30b pt.8/Hist1D_y1 (TH1F)  | scale label=0.001  applied=1000


In [10]:
zaxis = hists_pt_sim[0][1].GetZaxis()

In [15]:
hists_mass_ee = []
pt_ranges = [[0.0,0.5],[0.5,1.0],[1.0,1.5],[1.5,2.0],[2.0,2.5],[2.5,3.0],[3.0,4.0],[4.0,5.0]]
for i in range(5):
    hists_mass_pt = []
    for ipt in range(len(pt_ranges)):
        binlowpt = zaxis.FindBin(pt_ranges[ipt][0]+0.01)
        binhighpt = zaxis.FindBin(pt_ranges[ipt][1]-0.01)
        hists_mass_pt.append(hists_pt_sim[0][1+i].ProjectionY(f"hists_pt_sim_proj_ee{i}{ipt}",1,80,binlowpt,binhighpt))
        hists_mass_pt[-1].Scale(10.0 /50e6*93/20); hists_mass_pt[-1].Scale(1./hists_mass_pt[-1].GetBinWidth(1))
    hists_mass_ee.append(hists_mass_pt)


In [16]:
for ipt in range(len(pt_ranges)):
    for i in range(1,5):
        hists_mass_ee[0][ipt].Add(hists_mass_ee[i][ipt])

In [17]:
hists_mass_ee[0][0].Scale(10)

In [18]:
c12 = root.TCanvas("c12","c12",1200,600*4)
c12.Divide(2,4)
for ipt in range(len(pt_ranges)):
    c12.cd(ipt+1)
    root.gPad.SetLogy()
    hists_mass_ee[0][ipt].GetXaxis().SetRangeUser(0., 0.4)
    hists_mass_ee[0][ipt].Draw("HIST")
    objs_scaled_by_ipt[ipt+1].SetLineColor(root.kRed)
    objs_scaled_by_ipt[ipt+1].Draw("SAME HIST")
    #print ration between integrals in 0.01-0.1 GeV/c^2 maening the region where the cocktail is expected to dominate (we neeed toa ccount for bin width, since the histograms are in dN/dmee)
    integral_sim = hists_mass_ee[0][ipt].Integral(hists_mass_ee[0][ipt].FindBin(0.0), hists_mass_ee[0][ipt].FindBin(0.1))*hists_mass_ee[0][ipt].GetBinWidth(1)
    integral_data = objs_scaled_by_ipt[ipt+1].Integral(objs_scaled_by_ipt[ipt+1].FindBin(0.0), objs_scaled_by_ipt[ipt+1].FindBin(0.1))*objs_scaled_by_ipt[ipt+1].GetBinWidth(1)
    print(f"ipt={ipt}, integral_sim={integral_sim}, integral_data={integral_data}, ratio={integral_data/integral_sim if integral_sim > 0 else 0}")

c12.Draw()

ipt=0, integral_sim=0.00164151445517283, integral_data=0.005700000016950071, ratio=3.4724031817008494
ipt=1, integral_sim=0.00042813271390417494, integral_data=0.0008674499887274578, ratio=2.0261240511548744
ipt=2, integral_sim=0.0005653366184111676, integral_data=0.0006743000088317786, ratio=1.1927407262717984
ipt=3, integral_sim=0.0002628192221938905, integral_data=0.0002753599996503908, ratio=1.047716363178522
ipt=4, integral_sim=9.236380147789407e-05, integral_data=8.737499880226096e-05, ratio=0.9459874691620709
ipt=5, integral_sim=3.305515150532044e-05, integral_data=2.859700029148371e-05, ratio=0.8651299113507568
ipt=6, integral_sim=1.8301603016562106e-05, integral_data=1.4886500034663186e-05, ratio=0.8133986963432434
ipt=7, integral_sim=3.741590384115745e-06, integral_data=2.780000008897332e-06, ratio=0.7429995599463068
